# Philippine Address Pipeline — Improved Fuzzy-Match Edition

**Flow:** Raw Excel → Alias Normalization → N-gram Exact City/Province Match → WRatio Fuzzy Fallback → Province–City Cross-Validation → Strict Barangay Resolution → PSGC Code Mapping → Export

### Key improvements over previous version
| Issue | Old behaviour | Fix |
|---|---|---|
| Substring city false-match | `partial_token_set_ratio` → `TAYUM` matches `TAYUMAN` | Use `WRatio` for fuzzy; exact n-gram runs first |
| Incomplete city surface forms | Only matched dim keys like `CITY OF PARANAQUE` | `city_canonical` covers stripped prefix/suffix + all area_dict aliases + common abbrevs (`CSFP`, `CDO`, `CSJDM` …) |
| Accent mismatch | `DASMARIÑAS` ≠ `DASMARINAS` in lookup | Strip accents **before** alias expansion |
| No conflict resolution | City vs province conflict silently ignored | `cross_validate()` with prioritised rule set |
| Barangay over-match | Matched barangays across all cities | Gated to confirmed city via per-city index; strict ambiguity check |
| No PSGC from barangay | Code lookup was separate | `match_barangay()` returns `psgc_10_digit` directly |


## Cell 0 — Environment Check

In [19]:
import importlib

REQUIRED = {
    'pandas': 'pandas', 'numpy': 'numpy',
    'rapidfuzz': 'rapidfuzz', 'tqdm': 'tqdm',
    'openpyxl': 'openpyxl', 'xlsxwriter': 'xlsxwriter',
}
missing = []
for label, pkg in REQUIRED.items():
    try:
        mod = importlib.import_module(pkg)
        print(f'  OK  {label:<14} {getattr(mod, "__version__", "n/a")}')
    except ImportError:
        print(f'  !!  {label:<14} NOT FOUND')
        missing.append(pkg)
if missing:
    print(f'\npip install {" ".join(missing)}')
else:
    print('\nAll dependencies satisfied.')

  OK  pandas         3.0.1
  OK  numpy          2.4.3
  OK  rapidfuzz      3.14.3
  OK  tqdm           4.67.3
  OK  openpyxl       3.1.5
  OK  xlsxwriter     3.2.9

All dependencies satisfied.


## Cell 1 — Configuration

In [20]:
from pathlib import Path

BASE_DIR       = Path('..') / '..' / 'data'
INPUT_FILE     = str(BASE_DIR / 'sample'     / 'sample_address.xlsx')
DIM_LOCATION   = str(BASE_DIR / 'mapping'    / 'dim_location_20260316_v2.csv')
ALIAS_FILE     = str(BASE_DIR / 'utils'      / 'ph_address_alias_extended_v4.csv')
AREA_DICT_FILE = str(BASE_DIR / 'dictionary' / 'ph_area_dictionary_v2.json')
OUT_VERIFIED   = str(BASE_DIR / 'output'     / 'verified_addresses.xlsx')
OUT_INVALID    = str(BASE_DIR / 'output'     / 'invalid_addresses.xlsx')
OUT_COMBINED   = str(BASE_DIR / 'output'     / 'combined_addresses.xlsx')

# Fill paths here when processing multiple batch files
input_paths: list[Path] = []

# ── Matching thresholds (0-100) ───────────────────────────────────────────────
# City / Province: exact n-gram match always returns 100.
# Fuzzy fallback uses WRatio on individual tokens.
CITY_FUZZY_HIGH  = 88   # confident fuzzy match
CITY_FUZZY_LOW   = 80   # low-confidence fuzzy (still valid, lower confidence tier)
PROV_FUZZY_HIGH  = 90
PROV_FUZZY_LOW   = 82
BRGY_THRESHOLD   = 88   # partial_ratio minimum for barangay within confirmed city

MAX_ROWS: int | None = None   # None = process all

print('Config loaded.')

Config loaded.


## Cell 2 — Imports & Logging

In [21]:
import re, json, time, logging, unicodedata
from typing import Optional

import pandas as pd
import numpy as np
from IPython.display import display
from rapidfuzz import fuzz, process
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('ph_pipeline')
log.info('Imports OK')

23:56:47  INFO      Imports OK


## Cell 3 — Load Reference Data

Builds all lookup structures from the three reference files.

- **`city_canonical`** — every plausible surface form → dim city key (`CITY OF X`, `X CITY`, plain `X`, abbreviations, area_dict aliases)
- **`prov_canonical`** — province surface forms → dim province key
- **`_city_brgy_index`** — per-city list of `(brgy_norm, psgc)` for fast scoped barangay matching
- **`brgy_to_locations`** — barangay name → all `(city, prov, psgc)` tuples, used for ambiguity checking
- **`city_to_prov`** — city → allowed province(s), used in cross-validation

In [22]:
def _strip_accents(text: str) -> str:
    """NFD-decompose and drop combining marks → consistent ASCII comparison."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', str(text))
        if unicodedata.category(c) != 'Mn'
    )


def _read_csv(path: str) -> pd.DataFrame:
    for enc in ('utf-8-sig', 'utf-8', 'latin1', 'cp1252'):
        try:
            return pd.read_csv(path, encoding=enc)
        except (UnicodeDecodeError, Exception):
            continue
    raise ValueError(f'Cannot read: {path}')


def load_reference(dim_path: str, alias_path: str, area_dict_path: str):
    # ── dim_location ─────────────────────────────────────────────────────────
    dim = _read_csv(dim_path)
    dim['_city_norm'] = dim['city_name'].apply(lambda x: _strip_accents(x).upper())
    dim['_prov_norm'] = dim['province_name'].apply(lambda x: _strip_accents(x).upper())
    dim['_brgy_norm'] = dim['barangay_name'].apply(lambda x: _strip_accents(x).upper())
    log.info(f'dim_location: {len(dim):,} rows')

    # ── Alias CSV ────────────────────────────────────────────────────────────
    alias_df = _read_csv(alias_path)
    log.info(f'alias: {len(alias_df):,} pairs')

    # ── Area dictionary JSON ─────────────────────────────────────────────────
    with open(area_dict_path, encoding='utf-8-sig') as f:
        area_dict = json.load(f)
    log.info(f'area_dict v{area_dict.get("version", "?")} | cities: {len(area_dict.get("cities", {}))}')

    # ── Unique canonical lists ────────────────────────────────────────────────
    unique_cities = sorted(dim['_city_norm'].unique().tolist())
    unique_provs  = sorted(dim['_prov_norm'].unique().tolist())

    # ── city_canonical: surface form → dim city key ───────────────────────────
    city_canonical: dict[str, str] = {}

    for city in unique_cities:
        city_canonical[city] = city
        # Strip leading prefixes (longest first to avoid partial clobber)
        for prefix in ('CITY OF ', 'MUNICIPALITY OF ', 'MUNICIPAL CITY OF '):
            if city.startswith(prefix):
                short = city[len(prefix):].strip()
                city_canonical.setdefault(short, city)
        # Strip trailing suffixes
        for suffix in (' CITY', ' MUNICIPALITY'):
            if city.endswith(suffix):
                short = city[: -len(suffix)].strip()
                city_canonical.setdefault(short, city)

    # area_dict city + area aliases
    for city_key, meta in area_dict.get('cities', {}).items():
        ck = _strip_accents(city_key).upper()
        canonical = city_canonical.get(ck, ck)
        for alias in meta.get('aliases', []):
            city_canonical.setdefault(_strip_accents(alias).upper(), canonical)
        for area, area_meta in meta.get('areas', {}).items():
            if area_meta.get('infer_city_allowed', False):
                ak = _strip_accents(area).upper()
                city_canonical.setdefault(ak, canonical)
                for aa in area_meta.get('aliases', []):
                    city_canonical.setdefault(_strip_accents(aa).upper(), canonical)

    # Hard-coded abbreviations not covered by alias file
    _abbrevs = {
        'CSFP' : 'CITY OF SAN FERNANDO',   # Pampanga
        'CDO'  : 'CAGAYAN DE ORO',
        'CDOC' : 'CAGAYAN DE ORO',
        'ZC'   : 'ZAMBOANGA CITY',
        'DCN'  : 'DIGOS CITY',
        'QC'   : 'QUEZON CITY',
        'MNL'  : 'CITY OF MANILA',
        'BGC'  : 'CITY OF TAGUIG',
        'GSC'  : 'GENERAL SANTOS',
        'TC'   : 'CITY OF TARLAC',
        'CN'   : 'CAMARINES NORTE',
        'CSJDM': 'CITY OF SAN JOSE DEL MONTE',
        'PQUE' : 'CITY OF PARANAQUE',
    }
    for abbr, target in _abbrevs.items():
        resolved = city_canonical.get(target.upper(), target.upper())
        city_canonical.setdefault(abbr, resolved)

    # ── prov_canonical: surface form → dim province key ───────────────────────
    prov_canonical: dict[str, str] = {p: p for p in unique_provs}
    for prov_key in area_dict.get('province_city_map', {}):
        pk = _strip_accents(prov_key).upper()
        prov_canonical.setdefault(pk, pk)

    # ── brgy_to_locations: brgy_norm → [(city, prov, psgc)] ──────────────────
    brgy_to_locations: dict[str, list[tuple]] = {}
    for _, row in dim.iterrows():
        brgy_to_locations.setdefault(row['_brgy_norm'], []).append(
            (row['_city_norm'], row['_prov_norm'], str(row['psgc_10_digit']))
        )

    # ── city_to_prov ─────────────────────────────────────────────────────────
    city_to_prov: dict[str, list[str]] = {
        cn: grp['_prov_norm'].unique().tolist()
        for cn, grp in dim.groupby('_city_norm')
    }

    # ── Per-city barangay index ───────────────────────────────────────────────
    city_brgy_index: dict[str, list[tuple[str, str]]] = {}
    for _, row in dim.iterrows():
        city_brgy_index.setdefault(row['_city_norm'], []).append(
            (row['_brgy_norm'], str(row['psgc_10_digit']))
        )

    log.info(
        f'Lookups ready: {len(city_canonical)} city aliases | '
        f'{len(prov_canonical)} prov aliases | '
        f'{len(brgy_to_locations)} barangay keys | '
        f'{len(city_brgy_index)} cities indexed'
    )
    return (
        dim, alias_df, area_dict,
        unique_cities, unique_provs,
        city_canonical, prov_canonical,
        brgy_to_locations, city_to_prov,
        city_brgy_index,
    )


(
    dim, alias_df, area_dict,
    unique_cities, unique_provs,
    city_canonical, prov_canonical,
    brgy_to_locations, city_to_prov,
    city_brgy_index,
) = load_reference(DIM_LOCATION, ALIAS_FILE, AREA_DICT_FILE)

23:56:48  INFO      dim_location: 42,011 rows
23:56:48  INFO      alias: 512 pairs
23:56:48  INFO      area_dict v2.1.0 | cities: 158
23:56:56  INFO      Lookups ready: 1701 city aliases | 83 prov aliases | 26355 barangay keys | 1432 cities indexed


## Cell 4 — Stage 1: Alias Normalization

Applies the alias CSV (longest-match first), accent-stripped before regex so `Ñ → N` is uniform.

Order of operations:
1. `_strip_accents()` — uniform ASCII baseline
2. Remove `nan` literals (Excel blank cells)
3. Keep only `[A-Z0-9 ]`
4. Apply alias regex (longest pattern wins)
5. Remove consecutive duplicate tokens (`CITY CITY → CITY`)
6. Collapse whitespace

In [23]:
def build_alias_regex(alias_df: pd.DataFrame):
    pairs = [
        (str(p).strip(), str(r).strip())
        for p, r in zip(alias_df['pattern'], alias_df['replacement'])
        if str(p).strip() and str(r).strip()
    ]
    pairs.sort(key=lambda x: -len(x[0]))  # longest first → no partial clobber
    rmap = {p: r for p, r in pairs}
    pat  = '|'.join(re.escape(p) for p, _ in pairs)
    return re.compile(r'\b(' + pat + r')\b'), rmap


compiled_re, replace_map = build_alias_regex(alias_df)


def normalize(text: str) -> str:
    text = _strip_accents(str(text)).upper().strip()
    text = re.sub(r'\bNAN\b', ' ', text)         # Excel blank → remove
    text = re.sub(r'[^A-Z0-9\s]', ' ', text)      # punctuation → space
    text = compiled_re.sub(lambda m: replace_map[m.group(0)], text)
    text = re.sub(r'\b(\w+)( \1\b)+', r'\1', text) # duplicate tokens
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Smoke test
for raw in [
    'Salitran dasma cavite',
    '17 TOPAZ ST SEVERINA SUBD KM18 BRGY MARCELO GREEN PARANAQUE CITY',
    'Tagumpay sindalan CSFP',
    'BORACAY, ISLAND MALAY AKLAN',
    'sampalco manikla',
]:
    print(f'  {raw!r:55s} -> {normalize(raw)!r}')

  'Salitran dasma cavite'                                 -> 'SALITRAN DASMARIÑAS CITY CAVITE'
  '17 TOPAZ ST SEVERINA SUBD KM18 BRGY MARCELO GREEN PARANAQUE CITY' -> '17 TOPAZ STREET SEVERINA SUBDIVISION KM18 BARANGAY MARCELO GREEN nan CITY'
  'Tagumpay sindalan CSFP'                                -> 'TAGUMPAY SINDALAN CSFP'
  'BORACAY, ISLAND MALAY AKLAN'                           -> 'BORACAY ISLAND MALAY AKLAN'
  'sampalco manikla'                                      -> 'SAMPALCO MANIKLA'


## Cell 5 — Stage 2a: City Matching

**Pass 1 — N-gram exact lookup (window 6 → 1)**  
Slide over token sequence and check `city_canonical`. Prefers *longer* (more specific) matches.  
Catches multi-word cities (`SAN JOSE DEL MONTE`, `CAGAYAN DE ORO`) and abbreviations (`CSFP`, `CDO`, `QC`).

**Pass 2 — WRatio fuzzy token scan (only if pass 1 fails)**  
Each token ≥ 4 chars scored against `unique_cities` with `fuzz.WRatio`.  
`WRatio` penalises length mismatch — unlike `partial_token_set_ratio` it will **not** match `TAYUM` from `TAYUMAN`.

In [24]:
def match_city(norm_addr: str) -> tuple[Optional[str], int, str]:
    """
    Returns (dim_city_key | None, score, method).
    method: 'exact_ngram' | 'fuzzy_high' | 'fuzzy_low' | 'none'
    """
    tokens = norm_addr.split()

    # Pass 1 — exact n-gram (prefer longer matches)
    for size in range(min(6, len(tokens)), 0, -1):
        for i in range(len(tokens) - size + 1):
            phrase = ' '.join(tokens[i : i + size])
            if phrase in city_canonical:
                return city_canonical[phrase], 100, 'exact_ngram'

    # Pass 2 — WRatio fuzzy token scan
    best_m, best_s = None, 0
    for tok in tokens:
        if len(tok) < 4:
            continue
        r = process.extractOne(tok, unique_cities, scorer=fuzz.WRatio, score_cutoff=CITY_FUZZY_LOW)
        if r and r[1] > best_s:
            best_m, best_s = r[0], int(r[1])

    if best_m:
        method = 'fuzzy_high' if best_s >= CITY_FUZZY_HIGH else 'fuzzy_low'
        return best_m, best_s, method

    return None, 0, 'none'


# Validation probes
probes = [
    ('TAYUMAN MANILA',                                    'expect SAMPALOC or TONDO (not TAYUM)'),
    ('SALITRAN DASMARINAS CITY CAVITE',                   'expect CITY OF DASMARINAS'),
    ('BARANGAY MARCELO GREEN PARANAQUE CITY',              'expect CITY OF PARANAQUE'),
    ('TAGUMPAY SINDALAN CSFP',                            'expect CITY OF SAN FERNANDO'),
    ('TAPAZ CAPIZ',                                       'expect TAPAZ'),
    ('QUEZON CITY NCR',                                   'expect QUEZON CITY'),
    ('BLK 11 PEDRO NERI STREET RER SUBDIVISION KAUSWAGAN CDO', 'expect CAGAYAN DE ORO'),
    ('BORACAY ISLAND MALAY AKLAN',                        'expect MALAY'),
    ('KOREAN CRAVE UNITOP BALIUAG BULACAN',               'expect BALIWAG (BALIUAG alias)'),
    ('SOUTH CREST VILLAGE BARANGAY SAN AGUSTIN 2 DASMARINAS CITY CAVITE', 'expect CITY OF DASMARINAS'),
]
print(f'{"ADDRESS":<52} {"CITY":<28} SC  METHOD')
print('-' * 95)
for addr, note in probes:
    city, score, method = match_city(addr)
    print(f'{addr[:50]:<52} {str(city):<28} {score:>3}  {method}  # {note}')

ADDRESS                                              CITY                         SC  METHOD
-----------------------------------------------------------------------------------------------
TAYUMAN MANILA                                       ANILAO                        83  fuzzy_low  # expect SAMPALOC or TONDO (not TAYUM)
SALITRAN DASMARINAS CITY CAVITE                      CITY OF DASMARINAS           100  exact_ngram  # expect CITY OF DASMARINAS
BARANGAY MARCELO GREEN PARANAQUE CITY                CITY OF PARANAQUE            100  exact_ngram  # expect CITY OF PARANAQUE
TAGUMPAY SINDALAN CSFP                               CITY OF SAN FERNANDO         100  exact_ngram  # expect CITY OF SAN FERNANDO
TAPAZ CAPIZ                                          TAPAZ                        100  exact_ngram  # expect TAPAZ
QUEZON CITY NCR                                      QUEZON CITY                  100  exact_ngram  # expect QUEZON CITY
BLK 11 PEDRO NERI STREET RER SUBDIVISION KAUSWAGAN   

## Cell 6 — Stage 2b: Province Matching

Same two-pass strategy against `prov_canonical` / `unique_provs`.

In [25]:
def match_province(norm_addr: str) -> tuple[Optional[str], int, str]:
    """
    Returns (dim_prov_key | None, score, method).
    """
    tokens = norm_addr.split()

    # Pass 1 — exact n-gram
    for size in range(min(4, len(tokens)), 0, -1):
        for i in range(len(tokens) - size + 1):
            phrase = ' '.join(tokens[i : i + size])
            if phrase in prov_canonical:
                return prov_canonical[phrase], 100, 'exact_ngram'

    # Pass 2 — WRatio fuzzy token scan
    best_m, best_s = None, 0
    for tok in tokens:
        if len(tok) < 4:
            continue
        r = process.extractOne(tok, unique_provs, scorer=fuzz.WRatio, score_cutoff=PROV_FUZZY_LOW)
        if r and r[1] > best_s:
            best_m, best_s = r[0], int(r[1])

    if best_m:
        method = 'fuzzy_high' if best_s >= PROV_FUZZY_HIGH else 'fuzzy_low'
        return best_m, best_s, method

    return None, 0, 'none'

## Cell 7 — Stage 3: Province–City Cross-Validation

Cross-checks the independently matched city and province using `city_to_prov`.

**Conflict resolution priority:**
1. Both agree → `ok`
2. No province found → derive from city if city maps to exactly one province (`prov_derived`)
3. City exact (`score=100`) but province fuzzy → trust city, re-derive province (`conflict_city_kept`)
4. Province exact but city fuzzy → trust province, discard city (`conflict_prov_kept`)
5. Both fuzzy, conflict → keep higher score (`conflict_city_kept` / `conflict_prov_kept`)
6. No city found at all → `no_city` → address is **invalid**

In [26]:
def cross_validate(
    city: Optional[str], city_score: int, city_method: str,
    prov: Optional[str], prov_score: int, prov_method: str,
) -> tuple[Optional[str], Optional[str], str]:
    """
    Returns (resolved_city, resolved_prov, reason).
    reason: 'ok' | 'prov_derived' | 'city_only' |
            'conflict_city_kept' | 'conflict_prov_kept' | 'no_city'
    """
    if not city:
        return None, prov, 'no_city'

    allowed = city_to_prov.get(city, [])

    if not prov:
        if len(allowed) == 1:
            return city, allowed[0], 'prov_derived'
        return city, None, 'city_only'

    if prov in allowed:
        return city, prov, 'ok'

    # Conflict
    city_exact = (city_method == 'exact_ngram')
    prov_exact = (prov_method == 'exact_ngram')

    if city_exact and not prov_exact:
        resolved_prov = allowed[0] if len(allowed) == 1 else None
        return city, resolved_prov, 'conflict_city_kept'

    if prov_exact and not city_exact:
        return None, prov, 'conflict_prov_kept'

    # Both fuzzy — keep higher score
    if city_score >= prov_score:
        resolved_prov = allowed[0] if len(allowed) == 1 else None
        return city, resolved_prov, 'conflict_city_kept'
    else:
        return None, prov, 'conflict_prov_kept'


print('cross_validate() defined.')

cross_validate() defined.


## Cell 8 — Stage 4: Strict Barangay Resolution

Barangay matching is **only attempted after city is confirmed** (no conflict).

**Strictness rules:**
- Scores only barangays that exist **in the matched city** (via `city_brgy_index`) — eliminates cross-city false matches like `Poblacion` (553 cities!) being assigned wrongly
- Uses `partial_ratio` — good for finding a barangay name that is a substring of the full address
- Requires score ≥ `BRGY_THRESHOLD` (default 88)
- Final safety check: confirm the winning barangay actually exists for the matched city in `brgy_to_locations`
- Returns the original (accented) dim barangay name + `psgc_10_digit`

In [27]:
def match_barangay(
    norm_addr: str,
    city: str,
    prov: Optional[str],
) -> tuple[Optional[str], int, Optional[str]]:
    """
    Returns (barangay_display_name, score, psgc_10_digit).
    All three are None / 0 if no unambiguous match found.
    """
    entries = city_brgy_index.get(city, [])   # [(brgy_norm, psgc), ...]
    if not entries:
        return None, 0, None

    brgy_names = [e[0] for e in entries]
    result = process.extractOne(
        norm_addr, brgy_names,
        scorer=fuzz.partial_ratio,
        score_cutoff=BRGY_THRESHOLD,
    )
    if not result:
        return None, 0, None

    matched_brgy, score, idx = result
    psgc = entries[idx][1]

    # Safety: ensure barangay belongs to this city in dim
    owners = {loc[0] for loc in brgy_to_locations.get(matched_brgy, [])}
    if city not in owners:
        return None, 0, None

    # Retrieve original accented barangay name from dim
    mask = (dim['_city_norm'] == city) & (dim['_brgy_norm'] == matched_brgy)
    orig_rows = dim[mask]
    brgy_display = orig_rows['barangay_name'].iloc[0] if not orig_rows.empty else matched_brgy

    return brgy_display, int(score), psgc


print('match_barangay() defined.')

match_barangay() defined.


## Cell 9 — Stage 5: PSGC Code Lookup

Retrieves canonical names and codes from `dim_location`.  
If a barangay PSGC is already known (from `match_barangay`), uses it directly for the most specific row.  
Otherwise falls back to city+province level.

In [28]:
def lookup_codes(
    city: Optional[str],
    prov: Optional[str],
    psgc_from_brgy: Optional[str] = None,
) -> dict:
    empty = dict(
        psgc_10_digit=None, region_name=None, region_code=None,
        province_name=None, province_code=None,
        city_name=None, city_code=None,
        barangay_name=None, barangay_code=None,
    )

    if psgc_from_brgy:
        rows = dim[dim['psgc_10_digit'].astype(str) == psgc_from_brgy]
        if not rows.empty:
            r = rows.iloc[0]
            return dict(
                psgc_10_digit=psgc_from_brgy,
                region_name=r['region_name'], region_code=int(r['region_code']),
                province_name=r['province_name'], province_code=int(r['province_code']),
                city_name=r['city_name'], city_code=int(r['city_code']),
                barangay_name=r['barangay_name'], barangay_code=int(r['barangay_code']),
            )

    if not city:
        return empty

    subset = dim[dim['_city_norm'] == city]
    if prov:
        subset = subset[subset['_prov_norm'] == prov]
    if subset.empty:
        return empty

    r = subset.iloc[0]
    return dict(
        psgc_10_digit=None,
        region_name=r['region_name'], region_code=int(r['region_code']),
        province_name=r['province_name'], province_code=int(r['province_code']),
        city_name=r['city_name'], city_code=int(r['city_code']),
        barangay_name=None, barangay_code=None,
    )


print('lookup_codes() defined.')

lookup_codes() defined.


## Cell 10 — Main Orchestrator: `process_address()`

In [29]:
def process_address(raw: str) -> dict:
    out = dict(
        original_address=raw, address_normalized=None,
        barangay_name=None, barangay_code=None,
        city_name=None,     city_code=None,
        province_name=None, province_code=None,
        region_name=None,   region_code=None,
        psgc_10_digit=None,
        city_score=0, city_method='none',
        province_score=0, province_method='none',
        barangay_score=0,
        cross_validate_reason=None,
        is_valid=False, invalid_reason=None, confidence='none',
    )

    # Stage 1 — Normalize
    norm = normalize(raw)
    out['address_normalized'] = norm

    # Stage 2 — Match city and province independently
    city, cscore, cmethod = match_city(norm)
    prov, pscore, pmethod = match_province(norm)
    out.update(city_score=cscore, city_method=cmethod,
               province_score=pscore, province_method=pmethod)

    # Stage 3 — Cross-validate
    city, prov, xv = cross_validate(city, cscore, cmethod, prov, pscore, pmethod)
    out['cross_validate_reason'] = xv

    # Gate: city required
    if not city:
        out['is_valid']       = False
        out['invalid_reason'] = xv if xv != 'ok' else 'no_city'
        out['confidence']     = 'none'
        return out

    # Stage 4 — Strict barangay (city-scoped)
    brgy_display, bscore, psgc_brgy = match_barangay(norm, city, prov)
    out['barangay_score'] = bscore

    # Stage 5 — PSGC code lookup
    codes = lookup_codes(city, prov, psgc_brgy)
    out.update(
        psgc_10_digit  = codes['psgc_10_digit'],
        region_name    = codes['region_name'],
        region_code    = codes['region_code'],
        province_name  = codes['province_name'],
        province_code  = codes['province_code'],
        city_name      = codes['city_name'],
        city_code      = codes['city_code'],
        barangay_name  = codes['barangay_name'] or brgy_display,
        barangay_code  = codes['barangay_code'],
    )

    # Confidence tier
    if cscore == 100 and (pscore == 100 or xv in ('prov_derived', 'conflict_city_kept')):
        conf = 'high'
    elif cscore >= CITY_FUZZY_HIGH:
        conf = 'medium'
    else:
        conf = 'low'

    out.update(is_valid=True, invalid_reason=None, confidence=conf)
    return out


# Quick spot-check before running full batch
spot_checks = [
    'tapaz capiz',
    '17 TOPAZ ST SEVERINA DIAMOND SUBD KM18 BRGY MARCELO GREEN PARANAQUE CITY',
    'Salitran dasma cavite',
    'BORACAY, ISLAND MALAY AKLAN',
    'Tagumpay sindalan CSFP',
    'Gahonon, DCN',
    'BIR Bulua CDO',
    'sampalco manikla',
    'CENTRO ENRILE CAGAYAN',
    'FAMY, LAGUNA',
]
SPOT_COLS = ['original_address', 'city_name', 'province_name', 'barangay_name', 'confidence', 'cross_validate_reason', 'is_valid']
spot_df = pd.DataFrame([process_address(a) for a in spot_checks])
display(spot_df[SPOT_COLS])

,original_address,city_name,province_name,barangay_name,confidence,cross_validate_reason,is_valid
0,tapaz capiz,Tapaz,Capiz,NaN,high,ok,True
1,17 TOPAZ ST SEVERINA DIAMOND SUBD KM18 BRGY MA...,Batangas City,Batangas,NaN,medium,prov_derived,True
2,Salitran dasma cavite,City of Cavite,Cavite,NaN,high,ok,True
3,"BORACAY, ISLAND MALAY AKLAN",Malay,Aklan,NaN,high,ok,True
4,Tagumpay sindalan CSFP,City of San Fernando,Pampanga,Sindalan,high,prov_derived,True
5,"Gahonon, DCN",NaN,NaN,NaN,medium,city_only,True
6,BIR Bulua CDO,City of Cagayan De Oro,Misamis Oriental,Bulua,high,conflict_city_kept,True
7,sampalco manikla,Sampaloc,Metro Manila,NaN,low,ok,True
8,CENTRO ENRILE CAGAYAN,Enrile,Cagayan,NaN,high,ok,True
9,"FAMY, LAGUNA",Famy,Laguna,NaN,high,ok,True


## Cell 11 — Run Pipeline on All Rows

In [30]:
t_start = time.time()

if input_paths:
    existing = [p for p in input_paths if Path(p).exists()]
    df_input = pd.concat([pd.read_excel(p) for p in existing], ignore_index=True)
    log.info(f'Loaded {len(existing)} batch files → {len(df_input):,} rows')
else:
    df_input = pd.read_excel(INPUT_FILE)
    log.info(f'Loaded {INPUT_FILE} → {len(df_input):,} rows')

if MAX_ROWS:
    df_input = df_input.iloc[:MAX_ROWS]

addresses = df_input.iloc[:, 0].fillna('').astype(str).tolist()

records = [
    process_address(addr)
    for addr in tqdm(addresses, desc='Processing')
]

out_df  = pd.DataFrame(records)
elapsed = time.time() - t_start

n_valid   = int(out_df['is_valid'].sum())
n_invalid = len(out_df) - n_valid
log.info(f'Done in {elapsed:.1f}s — {n_valid:,}/{len(out_df):,} valid')

print(f'Valid   : {n_valid:,}  ({100*n_valid/len(out_df):.1f}%)')
print(f'Invalid : {n_invalid:,}  ({100*n_invalid/len(out_df):.1f}%)')
print('\nConfidence:')
print(out_df['confidence'].value_counts().to_string())
print('\nInvalid reasons:')
print(out_df['invalid_reason'].fillna('resolved').value_counts().to_string())
print('\nCross-validate reasons:')
print(out_df['cross_validate_reason'].value_counts().to_string())

23:56:57  INFO      Loaded ..\..\data\sample\sample_address.xlsx → 99 rows


Processing:   0%|          | 0/99 [00:00<?, ?it/s]

23:56:58  INFO      Done in 1.5s — 91/99 valid


Valid   : 91  (91.9%)
Invalid : 8  (8.1%)

Confidence:
confidence
high      51
medium    33
none       8
low        7

Invalid reasons:
invalid_reason
resolved              91
conflict_prov_kept     5
no_city                3

Cross-validate reasons:
cross_validate_reason
ok                    27
prov_derived          26
city_only             21
conflict_city_kept    17
conflict_prov_kept     5
no_city                3


## Cell 12 — Result Preview

In [31]:
DISPLAY_COLS = [
    'original_address', 'address_normalized',
    'barangay_name', 'city_name', 'province_name', 'region_name',
    'city_code', 'province_code', 'barangay_code', 'psgc_10_digit',
    'city_score', 'city_method', 'province_score', 'barangay_score',
    'confidence', 'cross_validate_reason', 'is_valid', 'invalid_reason',
]

verified_df = out_df[out_df['is_valid']].reset_index(drop=True)
invalid_df  = out_df[~out_df['is_valid']].reset_index(drop=True)

print(f'Verified rows: {len(verified_df):,}')
display(verified_df[DISPLAY_COLS].head(20))
print(f'\nInvalid rows: {len(invalid_df):,}')
display(invalid_df[DISPLAY_COLS].head(20))

Verified rows: 91


,original_address,address_normalized,barangay_name,city_name,province_name,region_name,city_code,province_code,barangay_code,psgc_10_digit,city_score,city_method,province_score,barangay_score,confidence,cross_validate_reason,is_valid,invalid_reason
0,Tagumpay sindalan CSFP,TAGUMPAY SINDALAN CSFP,Sindalan,City of San Fernando,Pampanga,Region III (Central Luzon),16.0,54.0,36.0,305416036,100,exact_ngram,0,100,high,prov_derived,True,NaN
1,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...",BLOCK 16 LOT 4 QUIOTAN STREET PHIL AM LIFE VIL...,Carmen,Carmen,Surigao del Sur,Region XIII (Caraga),6.0,68.0,3.0,1606806003,100,exact_ngram,0,92,medium,city_only,True,NaN
2,Cainta Rizal,CAINTA RIZAL,NaN,Cainta,Rizal,Region IV-A (CALABARZON),5.0,58.0,NaN,NaN,100,exact_ngram,100,0,high,ok,True,NaN
3,2079 a elias st sta cruz,2079 A ELIAS STREET SANTA CRUZ,NaN,Santa Cruz,Davao del Sur,Region XI (Davao Region),12.0,24.0,NaN,NaN,100,exact_ngram,0,0,medium,city_only,True,NaN
4,Swani Hardware 77 tirso neri street CDOC,SWANI HARDWARE 77 TIRSO NERI STREET CDOC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100,exact_ngram,0,0,medium,city_only,True,NaN
5,ADP Bldg Gov. A Lugay Ave Mangga Dos Matatalai...,ADP BUILDING GOV A LUGAY AVENUE MANGGA DOS MAT...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100,exact_ngram,0,0,medium,city_only,True,NaN
6,tapaz capiz,TAPAZ CAPIZ,NaN,Tapaz,Capiz,Region VI (Western Visayas),17.0,19.0,NaN,NaN,100,exact_ngram,100,0,high,ok,True,NaN
7,4662PagAsa St V mapa Sta Mesa Manila,4662PAGASA STREET V MAPA SANTA MESA MANILA,NaN,Santa,Ilocos Sur,Region I (Ilocos Region),22.0,29.0,NaN,NaN,100,exact_ngram,90,0,high,conflict_city_kept,True,NaN
8,CENTRO ENRILE CAGAYAN,CENTRO ENRILE CAGAYAN,NaN,Enrile,Cagayan,Region II (Cagayan Valley),12.0,15.0,NaN,NaN,100,exact_ngram,100,0,high,ok,True,NaN
9,873RizalStAlbayDistrict Bgy. 17 - Rizal Sreet....,873RIZALSTALBAYDISTRICT BARANGAY 17 RIZAL SREE...,Poblacion,Rizal,Cagayan,Region II (Cagayan Valley),21.0,15.0,23.0,201521023,100,exact_ngram,100,100,high,conflict_city_kept,True,NaN



Invalid rows: 8


,original_address,address_normalized,barangay_name,city_name,province_name,region_name,city_code,province_code,barangay_code,psgc_10_digit,city_score,city_method,province_score,barangay_score,confidence,cross_validate_reason,is_valid,invalid_reason
0,tayuman manila,TAYUMAN MANILA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83,fuzzy_low,90,0,none,conflict_prov_kept,False,conflict_prov_kept
1,CAGAYAN 3500 GOMEZ STREET,CAGAYAN 3500 GOMEZ STREET,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,90,fuzzy_high,100,0,none,conflict_prov_kept,False,conflict_prov_kept
2,ANA ROS VILLAGE MANDURRIAO,ANA ROS VILLAGE MANDURRIAO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,none,0,0,none,no_city,False,no_city
3,Brgy cabanutuan nueva ecija,BARANGAY CABANUTUAN NUEVA ECIJA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,90,fuzzy_high,100,0,none,conflict_prov_kept,False,conflict_prov_kept
4,BRGY FONZALES TANUAN CITY BATNGAS,BARANGAY FONZALES TANUAN CITY BATNGAS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,92,fuzzy_high,93,0,none,conflict_prov_kept,False,conflict_prov_kept
5,blk 11 lot 36 cerretos camela,BLOCK 11 LOT 36 CERRETOS CAMELA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,none,0,0,none,no_city,False,no_city
6,Westwoods Mandurriao,WESTWOODS MANDURRIAO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,none,0,0,none,no_city,False,no_city
7,Villa Iloillo City,VILLA ILOILLO CITY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,90,fuzzy_high,92,0,none,conflict_prov_kept,False,conflict_prov_kept


## Cell 13 — Export to Excel

In [32]:
out_root = Path(BASE_DIR) / 'output'
out_root.mkdir(parents=True, exist_ok=True)

batch_nums = [int(m.group(1)) for p in input_paths if (m := re.search(r'part_(\d+)', str(p).lower()))]
if batch_nums:
    suffix = f'parts_{min(batch_nums):03d}_{max(batch_nums):03d}'
elif input_paths:
    suffix = f'batch_{len(input_paths):03d}'
else:
    suffix = 'single'

BASE_COLS = [
    'original_address',
    'barangay_name', 'barangay_code',
    'city_name',     'city_code',
    'province_name', 'province_code',
    'region_name',   'region_code',
    'psgc_10_digit',
]
DIAG_COLS = BASE_COLS + [
    'address_normalized', 'confidence',
    'city_score', 'city_method',
    'province_score', 'province_method',
    'barangay_score',
    'cross_validate_reason', 'is_valid', 'invalid_reason',
]

def _write(df: pd.DataFrame, path: str, tab: str, color: str, cols: list):
    export_cols = [c for c in cols if c in df.columns]
    with pd.ExcelWriter(path, engine='xlsxwriter') as writer:
        df[export_cols].to_excel(writer, index=False, sheet_name=tab)
        ws = writer.sheets[tab]
        ws.set_tab_color(color)
        ws.autofit()
    log.info(f'Wrote {len(df):,} rows → {path}')


VF = str(out_root / f'verified_addresses_{suffix}.xlsx')
IF = str(out_root / f'invalid_addresses_{suffix}.xlsx')
CF = str(out_root / f'combined_addresses_{suffix}.xlsx')

_write(verified_df, VF, 'Verified', '#00B050', BASE_COLS)
_write(invalid_df,  IF, 'Invalid',  '#FF0000', BASE_COLS + ['address_normalized', 'invalid_reason', 'cross_validate_reason'])
_write(out_df,      CF, 'All',      '#0070C0', DIAG_COLS)

print(f'Verified  → {VF}')
print(f'Invalid   → {IF}')
print(f'Combined  → {CF}')

23:56:59  INFO      Wrote 91 rows → ..\..\data\output\verified_addresses_single.xlsx
23:56:59  INFO      Wrote 8 rows → ..\..\data\output\invalid_addresses_single.xlsx
23:56:59  INFO      Wrote 99 rows → ..\..\data\output\combined_addresses_single.xlsx


Verified  → ..\..\data\output\verified_addresses_single.xlsx
Invalid   → ..\..\data\output\invalid_addresses_single.xlsx
Combined  → ..\..\data\output\combined_addresses_single.xlsx


## Cell 14 — Full Pipeline Summary

In [33]:
total = len(out_df)
n_v   = int(out_df['is_valid'].sum())

print('═' * 70)
print('  PIPELINE SUMMARY')
print('═' * 70)
print(f'  Input rows          : {total:>8,}')
print(f'  Verified            : {n_v:>8,}  ({100*n_v/total:.1f}%)')
print(f'  Invalid             : {total-n_v:>8,}  ({100*(total-n_v)/total:.1f}%)')
print()
print('  Confidence (all rows):')
for conf, cnt in out_df['confidence'].value_counts().items():
    print(f'    {conf:<22} {cnt:>6,}  ({100*cnt/total:.1f}%)')
print()
print('  City-match method (valid rows):')
for meth, cnt in out_df.loc[out_df['is_valid'], 'city_method'].value_counts().items():
    print(f'    {meth:<26} {cnt:>6,}')
print()
print('  Cross-validate reasons:')
for reason, cnt in out_df['cross_validate_reason'].value_counts().items():
    print(f'    {reason:<30} {cnt:>6,}')
print()
brgy_filled = out_df['barangay_name'].notna().sum()
print(f'  Barangay resolved   : {brgy_filled:>8,}  ({100*brgy_filled/total:.1f}%)')
print(f'  PSGC (full 10-digit): {out_df["psgc_10_digit"].notna().sum():>8,}')
print(f'  Total elapsed       : {elapsed:.1f}s')
print('═' * 70)

══════════════════════════════════════════════════════════════════════
  PIPELINE SUMMARY
══════════════════════════════════════════════════════════════════════
  Input rows          :       99
  Verified            :       91  (91.9%)
  Invalid             :        8  (8.1%)

  Confidence (all rows):
    high                       51  (51.5%)
    medium                     33  (33.3%)
    none                        8  (8.1%)
    low                         7  (7.1%)

  City-match method (valid rows):
    exact_ngram                    77
    fuzzy_low                       7
    fuzzy_high                      7

  Cross-validate reasons:
    ok                                 27
    prov_derived                       26
    city_only                          21
    conflict_city_kept                 17
    conflict_prov_kept                  5
    no_city                             3

  Barangay resolved   :       15  (15.2%)
  PSGC (full 10-digit):       15
  Total elapsed       :

## Cell 15 — Invalid Pattern Diagnostics

In [34]:
invalid_only = out_df[~out_df['is_valid']].copy()
print(f'Invalid rows: {len(invalid_only):,}')
print('\nReasons:')
print(invalid_only['invalid_reason'].value_counts().to_string())
print()
for _, row in invalid_only[['original_address', 'address_normalized', 'invalid_reason', 'cross_validate_reason']].head(30).iterrows():
    print(f'  ORIG   : {row["original_address"]}')
    print(f'  NORM   : {row["address_normalized"]}')
    print(f'  REASON : {row["invalid_reason"]}  (xv: {row["cross_validate_reason"]})')
    print()

Invalid rows: 8

Reasons:
invalid_reason
conflict_prov_kept    5
no_city               3

  ORIG   : tayuman manila
  NORM   : TAYUMAN MANILA
  REASON : conflict_prov_kept  (xv: conflict_prov_kept)

  ORIG   : CAGAYAN 3500 GOMEZ STREET
  NORM   : CAGAYAN 3500 GOMEZ STREET
  REASON : conflict_prov_kept  (xv: conflict_prov_kept)

  ORIG   : ANA ROS VILLAGE MANDURRIAO
  NORM   : ANA ROS VILLAGE MANDURRIAO
  REASON : no_city  (xv: no_city)

  ORIG   : Brgy cabanutuan nueva ecija
  NORM   : BARANGAY CABANUTUAN NUEVA ECIJA
  REASON : conflict_prov_kept  (xv: conflict_prov_kept)

  ORIG   : BRGY FONZALES TANUAN CITY BATNGAS
  NORM   : BARANGAY FONZALES TANUAN CITY BATNGAS
  REASON : conflict_prov_kept  (xv: conflict_prov_kept)

  ORIG   : blk 11 lot 36 cerretos camela 
  NORM   : BLOCK 11 LOT 36 CERRETOS CAMELA
  REASON : no_city  (xv: no_city)

  ORIG   : Westwoods Mandurriao
  NORM   : WESTWOODS MANDURRIAO
  REASON : no_city  (xv: no_city)

  ORIG   : Villa Iloillo City
  NORM   : VILLA ILO

## Cell 16 — Reusable `run_pipeline()`

Convenience wrapper — call after any config change without re-running setup cells.

In [35]:
def run_pipeline(
    input_file     = INPUT_FILE,
    dim_path       = DIM_LOCATION,
    alias_path     = ALIAS_FILE,
    area_dict_path = AREA_DICT_FILE,
    out_verified   = OUT_VERIFIED,
    out_invalid    = OUT_INVALID,
    max_rows       = MAX_ROWS,
) -> pd.DataFrame:
    """Re-loads all reference data then processes input_file. Returns result DataFrame."""
    global dim, alias_df, area_dict
    global unique_cities, unique_provs
    global city_canonical, prov_canonical
    global brgy_to_locations, city_to_prov, city_brgy_index
    global compiled_re, replace_map

    (
        dim, alias_df, area_dict,
        unique_cities, unique_provs,
        city_canonical, prov_canonical,
        brgy_to_locations, city_to_prov,
        city_brgy_index,
    ) = load_reference(dim_path, alias_path, area_dict_path)
    compiled_re, replace_map = build_alias_regex(alias_df)

    df = pd.read_excel(input_file)
    if max_rows:
        df = df.iloc[:max_rows]
    addresses = df.iloc[:, 0].fillna('').astype(str).tolist()

    records = [process_address(a) for a in tqdm(addresses, desc='run_pipeline')]
    result  = pd.DataFrame(records)

    _write(result[result['is_valid']], out_verified, 'Verified', '#00B050', BASE_COLS)
    _write(result[~result['is_valid']], out_invalid, 'Invalid',  '#FF0000',
           BASE_COLS + ['address_normalized', 'invalid_reason', 'cross_validate_reason'])

    log.info(f'run_pipeline: {result["is_valid"].sum():,}/{len(result):,} valid')
    return result


# Example:
# result_df = run_pipeline(input_file='path/to/new_batch.xlsx')
print('run_pipeline() ready.')

run_pipeline() ready.
